In [1]:
import yaml
import pandas as pd
import os
import sys

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
sys.path.insert(0, project_root)

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
from functions.ann_model import KerasBinaryClassifier
from functions.evaluate_model import evaluate_model

In [4]:
from utils.utils import to_jsonl

In [5]:
# 1. Carregar configurações
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# pipeline selection    
with open(os.path.join("../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# model selection    
with open(os.path.join( "../config/model.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [6]:
# Get feature eng data
pipeline_name = "Pipeline2"

X_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_train_feat_eng_{pipeline_name}.parquet")
)

y_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_train_feat_eng_{pipeline_name}.parquet")
)

X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)

In [7]:
X_train

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize,numerical_pipe_trans__Fare,categorical_pipe__Pclass_1,categorical_pipe__Pclass_2,categorical_pipe__Sex_male,categorical_pipe__Embarked_S,categorical_pipe__Embarked_C,categorical_pipe__Title_Mr,categorical_pipe__Title_Miss,categorical_pipe__Cabin_1p_Rare
331,1.253641,-0.078684,1.0,0.0,3.384390,1,0,1,1,0,1,0,1
733,-0.477284,-0.377145,1.0,0.0,2.639057,0,1,1,1,0,1,0,0
382,0.215086,-0.474867,1.0,0.0,2.188856,0,0,1,1,0,1,0,0
704,-0.246494,-0.476230,0.0,0.1,2.180892,0,0,1,1,0,1,0,0
813,-1.785093,-0.025249,0.0,0.6,3.474293,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,-0.631144,-0.480162,1.0,0.0,2.157559,0,0,0,1,0,0,1,0
270,-0.092634,-0.030545,1.0,0.0,3.465736,1,0,1,1,0,1,0,0
860,0.907456,-0.355804,0.0,0.2,2.715244,0,0,1,1,0,1,0,0
435,-1.169653,1.683201,0.0,0.3,4.795791,1,0,0,1,0,0,1,1


In [9]:
X_val.drop(
    columns = config_model['voting_model']['cols_2_drop']['pipeline_2'],
    inplace=True
) 

X_train.drop(
    columns = config_model['voting_model']['cols_2_drop']['pipeline_2'],
    inplace=True
) 

In [10]:
X_train

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize,numerical_pipe_trans__Fare,categorical_pipe__Pclass_1,categorical_pipe__Pclass_2,categorical_pipe__Embarked_S,categorical_pipe__Embarked_C,categorical_pipe__Title_Mr,categorical_pipe__Title_Miss,categorical_pipe__Cabin_1p_Rare
331,1.253641,-0.078684,1.0,0.0,3.384390,1,0,1,0,1,0,1
733,-0.477284,-0.377145,1.0,0.0,2.639057,0,1,1,0,1,0,0
382,0.215086,-0.474867,1.0,0.0,2.188856,0,0,1,0,1,0,0
704,-0.246494,-0.476230,0.0,0.1,2.180892,0,0,1,0,1,0,0
813,-1.785093,-0.025249,0.0,0.6,3.474293,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
106,-0.631144,-0.480162,1.0,0.0,2.157559,0,0,1,0,0,1,0
270,-0.092634,-0.030545,1.0,0.0,3.465736,1,0,1,0,1,0,0
860,0.907456,-0.355804,0.0,0.2,2.715244,0,0,1,0,1,0,0
435,-1.169653,1.683201,0.0,0.3,4.795791,1,0,1,0,0,1,1


In [ ]:
from functions.cross_validate import cross_validate_kfold, cross_validate_StratifiedKFold, cross_validate_stratified_gs

In [ ]:
from functions.model_selection import grid_search_single_model_StratifiedKFold, grid_search_single_model_kfold

In [ ]:
from functions.single_model import SingleModelOrchestrator

In [ ]:
model_name = "LogisticRegression"

In [ ]:
model_orchestrator = SingleModelOrchestrator()
model_config = model_orchestrator.apply(model_name)  
    
param_grid = {
    "penalty":['l2'],
    "C":[10, 1, 0.1], 
    "max_iter":[200, 300, 1000], 
    "solver":['liblinear', 'saga', 'lbfgs']    
    }     


In [ ]:
best_paramns = grid_search_single_model_StratifiedKFold(
        X_train, 
        y_train, 
        model_config['model'], 
        param_grid, 
        metric='roc_auc'
        )     

In [ ]:

# 4. train model 
    # set params
model_clf = model_config['model'].set_params(**best_paramns)


In [ ]:
    # cross-validade
df_cv = cross_validate_StratifiedKFold(
    X_train, 
    y_train, 
    model_clf,
    model_config
    )

In [ ]:
df_cv

In [ ]:
model = KerasBinaryClassifier(input_dim=X_train.shape[1], epochs=70)

clf = model.fit(X_train, y_train)

In [ ]:
metrics_train = evaluate_model(clf, X_train, y_train)

In [ ]:
metrics_val = evaluate_model(clf, X_val, y_val)

In [ ]:
metrics_train['accuracy_score']

In [ ]:
metrics_val['accuracy_score']

In [ ]:
pipeline = make_pipeline(
    KerasBinaryClassifier(input_dim=X_train.shape[1])
)

pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "kerasbinaryclassifier__epochs": [70, 90],
    "kerasbinaryclassifier__batch_size": [16, 32],
    "kerasbinaryclassifier__hidden_units": [(32,16), (64,32), (32,32)],
    "kerasbinaryclassifier__learning_rate": [0.001, 0.0005]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
grid.best_params_

In [ ]:
metrics_train_grid = evaluate_model(grid, X_train, y_train)

In [ ]:
metrics_train_grid['accuracy_score']

In [ ]:
metrics_val_grid = evaluate_model(grid, X_val, y_val)

In [ ]:
metrics_val_grid['accuracy_score']